In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('C:\\Users\\h1n2e\\Downloads\\airtel_data\\Airtel_telecom_analytics\\Airtel_telecom_analytics\\data\\raw\\airtel.db')
def q(sql):
    return pd.read_sql(sql, con)

In [2]:
# A1 · How many subscribers are prepaid and 
# how many postpaid, and what share is each?
# STEPS
# 1. Open customer_features, where one row is one subscriber.
# 2. Group by connection_type.
# 3. Count the rows in each group.
# 4. Express each count as a share of 22,000. 

q("""
SELECT
    connection_type,
    COUNT(*) AS subscriber_count,
    ROUND((COUNT(*) * 100.0 / 22000), 2) AS perc
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subscriber_count,perc
0,Postpaid,4851,22.05
1,Prepaid,17149,77.95


In [3]:
# A2 · Which five plans have the most subscribers, and what does each cost?
# STEPS
# 1. Group customer_features by plan_name.
# 2. Carry plan_type and price_inr through the group.
# 3. Count and sort, biggest first.
# 4. Show the top five.
q("""
SELECT plan_name, plan_type, price_inr, COUNT(*) AS subs
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY subs DESC
LIMIT 5;
""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [4]:
# A3 · Which five circles have the most subscribers?
# STEPS
# 1. Group by circle_name, carrying region.
# 2. Count.
# 3. Sort and take five.
q("""
SELECT circle_name, COUNT(*) AS subs
FROM customer_features
GROUP BY circle_name
ORDER BY subs DESC
LIMIT 5;
""")

,circle_name,subs
0,Maharashtra,1428
1,Andhra Pradesh,1412
2,Mumbai,1395
3,Karnataka,1361
4,North East,1318


In [5]:
# A4 · How many subscribers are in each age band?
# STEPS
# 1. Group by age_band.
# 2. Count and express as a share.
# 3. Sort by band, not by size, so the shape of the distribution is visible.
q("""
SELECT age_band, COUNT(*) AS subs,
ROUND(100.0*COUNT(*)/22000) AS perc
FROM customer_features
GROUP BY age_band
ORDER BY subs DESC;
""")

,age_band,subs,perc
0,25-34,7467,34.0
1,18-24,5268,24.0
2,35-44,4824,22.0
3,45-54,2895,13.0
4,55+,1546,7.0


In [6]:
# A5 · How many subscribers did each acquisition channel bring in?
# STEPS
# 1. Group by acquisition_channel.
# 2. Count and share.
# 3. Sort biggest first.
q("""
SELECT acquisition_channel, COUNT(*) AS subs,
ROUND(100.0*COUNT(*)/22000) AS perc
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC;
""")

,acquisition_channel,subs,perc
0,Retailer,9287,42.0
1,Airtel App,6513,30.0
2,Website,3081,14.0
3,Call Center,1770,8.0
4,Doorstep,1349,6.0


In [7]:
# A6 · How many subscribers use each device type and operating system?
# STEPS
# 1. Group by device_type and os together.
# 2. Count.
# 3. Read the result and notice that only three of the
# six possible combinations exist.
q("""
SELECT device_type, os, COUNT(*) AS subs
FROM customer_features
GROUP BY device_type, os
ORDER BY subs DESC;
""")

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


In [8]:
 # A7 · How many subscribers have 0, 1, 2 and 3 completed years with us?
# S T E P S
# 1. Group by tenure_years, which Step 2 calculated from activation_date.
# 2. Count and share.
# 3. Sort by year.
q("""
SELECT 
    tenure_years,
    COUNT(*) AS subs,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS perc
FROM customer_features
GROUP BY tenure_years
ORDER BY subs DESC;
""")

,tenure_years,subs,perc
0,2,7055,32.07
1,1,6961,31.64
2,0,4610,20.95
3,3,3374,15.34


In [9]:
# B1 · What is our average monthly revenue per subscriber,
# by the company's ARPU figure? — 

# STEPS
# 1. Open customer_features.
# 2. Average stated_arpu.
# 3. Report it.
q("""
SELECT  round(AVG(stated_arpu), 2) AS avg_arpu
FROM customer_features
""")

,avg_arpu
0,421.65


In [10]:
# B2 · What is our average monthly revenue per subscriber, from the payments received?
# STEPS
# 1. Open customer_features again.
# 2. Average actual_arpu_6m, which Step 2 built from recharges.
# 3. Also total it, to get the monthly revenue of the whole base.

q("""

select ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
SUM(actual_arpu_6m) AS total_monthly_revenue
from customer_features;


""")

,avg_actual_arpu,total_monthly_revenue
0,344.46,7578210.14


In [11]:
# B3 · By how much does the company's ARPU figure overstate the money received?
# STEPS
# 1. Average both columns in one query.
# 2. Average the gap.
# 3. Express the difference as a percentage of the actual total, not of the stated one.\
q("""
select ROUND(AVG(stated_arpu), 2) AS stated_arpu,
ROUND(AVG(actual_arpu_6m), 2) AS actual_revenue,
ROUND(AVG(stated_arpu - actual_arpu_6m), 2) AS gap,
ROUND(100.0 * SUM(stated_arpu - actual_arpu_6m) / SUM(actual_arpu_6m), 2) AS perc_overstatement
from customer_features;
""")

,stated_arpu,actual_revenue,gap,perc_overstatement
0,421.65,344.46,77.18,22.41


In [12]:
# B4 · Is that gap the same for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Average both columns inside each group.
# 3. Read the two rows against each other rather than down the column.
q("""
select connection_type, 
count(*) AS subs,    
ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu
from customer_features
group by connection_type;  
""")

,connection_type,subs,avg_stated_arpu,avg_actual_arpu
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


In [13]:
# B5 · Which five plans bring in the most actual monthly revenue?
# STEPS
# 1. Group by plan_name.
# 2. Sum actual_arpu_6m for total monthly revenue and average it for revenue per subscriber.
# 3. Sort by total, take five.
q("""
SELECT plan_name,
    count(*) AS subs,
    SUM(actual_arpu_6m) AS total_monthly_revenue,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_revenue_per_subscriber
FROM customer_features
GROUP BY plan_name 
ORDER BY total_monthly_revenue DESC
LIMIT 5;
""")

,plan_name,subs,total_monthly_revenue,avg_revenue_per_subscriber
0,Postpaid Infinity 1499,1431,2171756.20,1517.65
1,Postpaid Family 999,1239,1249245.94,1008.27
2,Postpaid 749,946,718958.94,760.00
3,Prepaid Entertainment 399,2142,671849.50,313.66
4,Prepaid Data 349,1884,520066.00,276.04


In [14]:
# B6 · Which five circles bring in the most actual monthly revenue?
# STEPS
# 1. Group by circle_name.
# 2. Sum actual_arpu_6m.
# 3. Sort and take five.
q("""
SELECT circle_name, 
    count(*) AS subs,
    SUM(actual_arpu_6m) AS total_monthly_revenue,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_revenue_per_subscriber
FROM customer_features
GROUP BY circle_name
ORDER BY total_monthly_revenue DESC
LIMIT 5;
""")

,circle_name,subs,total_monthly_revenue,avg_revenue_per_subscriber
0,Mumbai,1395,473288.16,339.27
1,Karnataka,1361,468514.95,344.24
2,Andhra Pradesh,1412,463369.80,328.17
3,North East,1318,460998.29,349.77
4,Maharashtra,1428,450032.30,315.15


In [15]:
# B7 · How much money did we receive from January to June 2025?
# STEPS
# 1. This is a question about payments, not subscribers, so read recharges.
# 2. Filter to the window.
# 3. Count and sum.
q("""   
SELECT COUNT(*) AS total_recharges,
    SUM(amount_inr) AS total_revenue
FROM recharges
WHERE recharge_date >= '2025-01-01' AND recharge_date <= '2025-06-30';
""")

,total_recharges,total_revenue
0,83693,45469268


In [16]:
# B8 · Do subscribers who have been with us longer pay more each month?
# STEPS
# 1. Group by tenure_years.
# 2. Average actual_arpu_6m.
# 3. Sort by year

q("""

select tenure_years,
count(*) AS subs,
round(AVG(actual_arpu_6m), 2) AS avg_monthly_revenue_per_subscriber
from customer_features
group by tenure_years;

""")

,tenure_years,subs,avg_monthly_revenue_per_subscriber
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


In [17]:
# C1 · How many payments of each type did we take, and from how many subscribers?
# STEPS
# 1. Read recharges, where one row is one payment.
# 2. Group by txn_type.
# 3. Count rows and distinct subscribers in each group.
q("""
SELECT txn_type, 
       COUNT(DISTINCT customer_id) AS subs,
       COUNT(*) AS payments
FROM recharges
GROUP BY txn_type;
""")

,txn_type,subs,payments
0,Bill Payment,4851,108419
1,Recharge,17149,299670


In [18]:
# C2 · What is the average value of a payment? — deliberate failure 2

# STEPS
# 1. Read recharges.
# 2. Average amount_inr.
q("""
select round(AVG(amount_inr), 2) AS avg_payment_value
from recharges;
""")

,avg_payment_value
0,532.59


In [19]:
# C3 · What is the average payment for a prepaid recharge, and for a postpaid bill?
# STEPS
# 1. Same table, same average.
# 2. Add GROUP BY txn_type.
# 3. Add the minimum and maximum so the two ranges can be compared.
q("""
select txn_type, 
       round(AVG(amount_inr), 2) AS avg_payment_value,
       round(MIN(amount_inr), 2) AS min_payment_value,
       round(MAX(amount_inr), 2) AS max_payment_value
from recharges
group by txn_type;
""")

,txn_type,avg_payment_value,min_payment_value,max_payment_value
0,Bill Payment,1082.42,399.0,1873.0
1,Recharge,333.66,99.0,1799.0


In [20]:
# C4 · How many times did a prepaid and a postpaid subscriber pay in six months?
# S T E P S
# 1. Back to customer_features, because this is now a question about subscribers.
# 2. Group by connection_type.
# 3. Average recharge_count and recharge_total_6m.

q("""

select connection_type,sum(recharge_count) AS total_payments,
round(AVG(recharge_count), 2) AS avg_payments_per_subscriber,
sum(recharge_total_6m) AS total_revenue,
round(AVG(recharge_total_6m), 2) AS avg_revenue_per_subscriber
from customer_features
group by connection_type;


""")

,connection_type,total_payments,avg_payments_per_subscriber,total_revenue,avg_revenue_per_subscriber
0,Postpaid,26156,5.39,28244989,5822.51
1,Prepaid,57537,3.36,17224279,1004.39


In [21]:
# C5 · Which payment channels take the most payments?
# STEPS
# 1. Read recharges.
# 2. Group by channel.
# 3. Count and share.
q("""
select channel, 
       COUNT(*) AS payments,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 2) AS perc
from recharges
group by channel
order by payments desc;

""")

,channel,payments,perc
0,Website,111163,27.24
1,Airtel App,111083,27.22
2,USSD,75257,18.44
3,Retailer,74406,18.23
4,AutoPay,36180,8.87


In [22]:
# C6 · Do prepaid and postpaid subscribers pay through the same channels?
# STEPS
# 1. Group by txn_type and channel.
# 2. Use a window function to express each count as a share of its own txn_type, so the two groups can be 
# compared fairly.
q("""
select txn_type, channel, COUNT(*) AS payments,
ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY txn_type), 2) AS perc
from recharges
group by txn_type, channel
order by txn_type, perc desc;
""")

,txn_type,channel,payments,perc
0,Bill Payment,Airtel App,36186,33.38
1,Bill Payment,AutoPay,36180,33.37
2,Bill Payment,Website,36053,33.25
3,Recharge,USSD,75257,25.11
4,Recharge,Website,75110,25.06
5,Recharge,Airtel App,74897,24.99
6,Recharge,Retailer,74406,24.83


In [23]:
# C7 · Does the number of payments a subscriber made go with whether they left?
# STEPS
# 1. Group customer_features by recharge_count.
# 2. Count subscribers and average churn in each group.
# 3. Restrict to counts of eight or fewer, where the groups are large enough to read.
q("""
select recharge_count,COUNT(*) AS subs,
ROUND(100 * AVG(is_churned), 2) AS avg_churn
from customer_features
group by recharge_count
having recharge_count <= 8
order by recharge_count;

""")

,recharge_count,subs,avg_churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


In [24]:
#D1 · How much data did the network carry each month, and for how many subscribers?
#STEPS
#1. Read v_usage_month, where one row is one subscriber in one month.
#2. Group by month.
#3. Sum the data and convert megabytes to terabytes.
#4. Also count the distinct subscribers who appear, because that number is going to matter.
q("""

select month, 
count(*)AS subs,
round(sum(total_mb)/1024/1024, 2) AS total_tb
from v_usage_month
group by month
order by month; 


""")

,month,subs,total_tb
0,2025-01,21302,337.97
1,2025-02,21162,307.32
2,2025-03,20366,332.51
3,2025-04,19320,310.27
4,2025-05,18356,363.10
5,2025-06,17326,338.95


In [25]:
#  D2 · What is a subscriber's average daily data use?
# STEPS
# 1. Read usage_daily.
# 2. Average data_mb.
# 3. Report it.
q("""

select round(AVG(data_mb), 2) AS avg_daily_data_use
from usage_daily;


""")

,avg_daily_data_use
0,1572.87


In [26]:
#D3 · What is the average data per subscriber, per active day and per observed day?
#STEPS
#1. Read customer_features, where Step 2 already computed both.
#2. Average each.
#3. Divide one by the other.
q("""
SELECT 
    ROUND(avg(mb_per_observed_day), 2) AS avg_obsrv_day,
    ROUND(avg(mb_per_active_day), 2) AS avg_active_day 
FROM customer_features;
""")

,avg_obsrv_day,avg_active_day
0,524.05,1404.44


In [27]:
#D4 · Does data use per day change with the day of the week?
#STEPS
#1. Read usage_daily.
#2. Use strftime('%w', usage_date) to pull the weekday out.
#3. Group by it.
#4. Count rows and average the data.
q("""
select strftime('%w', usage_date) AS weekday,
count(*) AS subs,                                           
round(AVG(data_mb), 2) AS avg_daily_data_use
from usage_daily
group by weekday
order by weekday;           
""")

,weekday,subs,avg_daily_data_use
0,0,190368,1578.06
1,1,189914,1572.94
2,2,183823,1569.55
3,3,191088,1564.48
4,4,190764,1580.00
5,5,190542,1569.51
6,6,190240,1575.45


In [28]:
#D5 · How many subscribers used the network in each month?
#STEPS
#1. Read v_usage_month.
#2. Count distinct subscribers per month.
#3. Read the trend rather than any single number.
q("""
select month,
count(distinct customer_id) AS subs
from v_usage_month
group by month
""")

,month,subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [29]:
#D6 · Did voice minutes and SMS per day change from January to June?
#STEPS 
#1. Read usage_daily. 
#2. Bucket by month with strftime.
#3. Average voice_minutes a
# nd sms_count.
q("""
select strftime('%m', usage_date) AS month,
avg(voice_minutes) AS avg_voice_minutes,
avg(sms_count) AS avg_sms_count
from usage_daily
group by month
""")

,month,avg_voice_minutes,avg_sms_count
0,01,55.559134,0.745112
1,02,55.841680,0.746221
2,03,56.080162,0.750497
3,04,56.485626,0.760088
4,05,56.700352,0.764229
5,06,57.125820,0.771955


In [30]:
#D7 · How much data did an active subscriber use each month, and on how many days?
#STEPS
#1. Read v_usage_month.
#2. Group by month.
#3. Average total_mb per subscriber and convert to gigabytes.
#4. Also average usage_days, to see whether people are appearing more often or using more when they appear.
q("""
select month,
round(avg(total_mb)/1024, 2) as avg_data_gb,
round(avg(usage_days), 2) as avg_usage_days
from v_usage_month
group by month
""")

,month,avg_data_gb,avg_usage_days
0,2025-01,16.25,11.39
1,2025-02,14.87,10.31
2,2025-03,16.72,11.52
3,2025-04,16.44,11.25
4,2025-05,20.26,11.73
5,2025-06,20.03,11.46


In [31]:
#D8 · Among subscribers who never left, did data per active day still rise in May? STEPS
# 1. The base changes composition every month as people leave, so hold it constant. 
#2. Join v_usage_month to customer_features and keep only subscribers who never churned.
#3. Average data per active day within each month.
q("""
SELECT m.month, COUNT(*) AS subs,
ROUND(AVG(m.total_mb/m.usage_days),1) AS mb_per_day
FROM v_usage_month m
JOIN customer_features c ON c.customer_id = m.customer_id
WHERE c.is_churned=0
GROUP BY m.month
ORDER BY m.month;
""")

,month,subs,mb_per_day
0,2025-01,16293,1410.6
1,2025-02,16588,1422.1
2,2025-03,16640,1418.1
3,2025-04,16648,1411.8
4,2025-05,16667,1659.5
5,2025-06,16660,1662.2


In [32]:
#E1 · Split into four equal groups by daily data, what range does each group cover?
#STEPS
#1. Take customer_features, excluding the four subscribers with no usage at all.
#2. Apply NTILE(4) ordered by mb_per_active_day.
#3. Group by the quartile and report its size and range.
q("""
WITH n AS (SELECT mb_per_active_day AS m,
                  NTILE(4) OVER (ORDER BY mb_per_active_day) as q
           FROM customer_features
           WHERE usage_days > 0)

SELECT q, COUNT(*) AS subs,
       ROUND(MIN(m),0) AS mn,
       ROUND(MAX(m),0) AS mx
FROM n GROUP BY q
ORDER BY q;
""")

,q,subs,mn,mx
0,1,5499,25.0,657.0
1,2,5499,657.0,1093.0
2,3,5499,1093.0,1572.0
3,4,5499,1572.0,9720.0


In [33]:
#Do heavy data users leave less often than light ones?
#STEPS
#1. Same quartiles.
#2. Carry is_churned through the CTE.
#3. Average it per quartile.
q("""
   WITH n AS (
    SELECT is_churned,
           NTILE(4) OVER (ORDER BY mb_per_active_day) AS q
    FROM customer_features
    WHERE usage_days > 0
)
SELECT q, 
       COUNT(*) AS subs,
       ROUND(100 * AVG(is_churned), 2) AS churned
FROM n 
GROUP BY q
ORDER BY q;
""")

,q,subs,churned
0,1,5499,30.28
1,2,5499,28.11
2,3,5499,22.51
3,4,5499,14.09


In [34]:
#Does voice use per day differ by connection type and device, as data use does?
#STEPS
#1. Group by connection_type and device_type.
#2. Compute voice minutes per active day and megabytes per active day side by side.
#3. Sort by the voice figure.
q("""
 SELECT connection_type, device_type, COUNT(*) AS subs,
 ROUND(AVG(total_voice_min*1.0)/NULLIF(usage_days,0),2) AS voice_per_day,
 ROUND(AVG(mb_per_active_day),1) AS mb_per_day
FROM customer_features
WHERE usage_days>0
GROUP BY connection_type, device_type
ORDER BY voice_per_day DESC;


""")

,connection_type,device_type,subs,voice_per_day,mb_per_day
0,Prepaid,Feature Phone,2338,54.63,166.4
1,Prepaid,Smartphone,14807,51.62,1093.1
2,Postpaid,Smartphone,4161,30.58,3352.8
3,Postpaid,Feature Phone,690,27.19,530.7


In [35]:
#E4 · For each plan, how much of the monthly data allowance does a subscriber use?
#STEPS
#1. Group by plan_name, carrying the monthly_gb allowance from v_plans_clean.
#2. Convert each subscriber's six-month data total to gigabytes per month.
#3. Average it per plan.
#4. Sort by allowance so the two columns can be read against each other.
q("""
 SELECT plan_name, monthly_gb,
 ROUND(AVG(total_data_mb) / 1024, 2) AS avg_gb
 FROM customer_features
 WHERE usage_days > 0
 GROUP BY plan_name, monthly_gb
 ORDER BY monthly_gb;
""")


,plan_name,monthly_gb,avg_gb
0,Prepaid Lite 99,15.0,17.31
1,Prepaid Smart 155,30.0,34.31
2,Prepaid Value 209,30.0,35.33
3,Postpaid 399,40.0,47.13
4,Prepaid Max 479,45.0,53.06
5,Prepaid Popular 299,45.0,53.59
6,Prepaid Annual 1799,60.0,71.22
7,Prepaid Quarterly 719,60.0,70.67
8,Postpaid 499,75.0,92.82
9,Prepaid Data 349,75.0,90.76


In [36]:
#Do the subscribers who spend the most also use the most data?
#STEPS
#1. Group customer_features by m_score, the monetary quartile built in Step 2.
#2. Average spend, data per active day and churn together.
q("""
 SELECT m_score, COUNT(*) AS subs,
ROUND(AVG(recharge_total_6m),0) AS spended,
ROUND(AVG(mb_per_active_day),0) AS mb_per_day,
ROUND(100*AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY m_score
ORDER BY m_score;
""")


,m_score,subs,spended,mb_per_day,churn
0,1,5500,167.0,973.0,54.82
1,2,5500,945.0,779.0,19.96
2,3,5500,1678.0,1117.0,11.24
3,4,5500,5477.0,2750.0,9.00


In [37]:
#· How many SMS does a subscriber send per day, and how many never send one?
#STEPS
#1. Average SMS per active day across everybody with usage.
#2. Count how many never sent one at all.
q("""
  SELECT 
    ROUND(AVG(total_sms * 1.0 / NULLIF(usage_days, 0)), 2) AS sms_per_day,
    SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_used,
    COUNT(*) AS subs
FROM customer_features
WHERE usage_days > 0;
""")




,sms_per_day,never_used,subs
0,0.6,654,21996


In [38]:
#F1 · How common is roaming, counted in days and in subscribers?
#STEPS
#1. Read usage_daily, where one row is one subscriber-day.
#2. Group by is_roaming.
#3. Count days, express as a share, and count distinct subscribers.
q("""
 SELECT is_roaming, COUNT(*) AS days,
ROUND(100*COUNT(*)/1326739,2) AS perc,
COUNT(DISTINCT customer_id) AS subs
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,days,perc,subs
0,0,1287143,97.0,21994
1,1,39596,2.0,16233


In [39]:
#F2 · Do subscribers use more or less on roaming days than at home?
#STEPS
#1. Same table, same grouping.
#2. Average data_mb and voice_minutes in each group.
#3. Read the two rows against each other.
q("""
  SELECT is_roaming,
       ROUND(AVG(voice_minutes), 2) AS voice_min,
       ROUND(AVG(data_mb), 2) AS data
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,voice_min,data
0,0,56.27,1572.95
1,1,56.09,1570.05


In [40]:
#F3 · Do postpaid subscribers roam more than prepaid ones?
#STEPS
#1. Move to customer_features, where roaming_days is already counted per subscriber.
#2. Group by connection_type.
#3. Average the roaming days and count what share ever roamed at all.
q("""
 SELECT connection_type, COUNT(*) AS subs,
ROUND(AVG(roaming_days), 2) AS avg_roaming,
ROUND((100 * AVG(roaming_days > 0)), 2) AS total_perc
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,avg_roaming,total_perc
0,Postpaid,4851,1.84,74.83
1,Prepaid,17149,1.79,73.49


In [41]:
#F5 · Which five circles roam the most?
#STEPS
#1. Group customer_features by circle.
#2. Divide total roaming days by total usage days, so circles of different sizes are comparable.
#3. Sort and take five.
q("""
SELECT circle_name,
ROUND(100.0*SUM(roaming_days)/SUM(usage_days),2) AS perc
FROM customer_features
GROUP BY circle_name
ORDER BY perc DESC;
""")

,circle_name,perc
0,Odisha,3.07
1,Haryana,3.07
2,Delhi,3.07
3,Karnataka,3.05
4,Assam,3.05
5,Uttar Pradesh (East),3.03
6,Maharashtra,3.03
7,Himachal Pradesh,3.03
8,West Bengal,3.02
9,Gujarat,3.00


In [42]:
# F4 · Do subscribers who roam more often leave less often?
# STEPS
# 1. Band roaming_days into four groups with CASE WHEN.
# 2. Count and average churn in each.
# 3. Read the gradient — and then be suspicious of it.
q("""
SELECT CASE WHEN roaming_days=0 THEN '0'
            WHEN roaming_days<=2 THEN '0-2'
            WHEN roaming_days<=5 THEN '3-5'
            ELSE '6+'
       END AS band,
       COUNT(*) AS subs,
       ROUND(100.0*AVG(is_Churned),2) AS churn
FROM customer_features
WHERE usage_days>0
GROUP BY band
ORDER BY band;
""")

,band,subs,churn
0,0,5763,44.61
1,0-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


In [43]:
# G1 · What do subscribers complain about most?
# STEPS
# 1. Read complaints, where one row is one ticket.
# 2. Group by category.
# 3. Count and share.
# 4. Sort biggest first.
q("""
SELECT category, COUNT(*) AS tickets,
ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM complaints),2) AS perc
FROM complaints
GROUP BY category
ORDER BY tickets DESC;
""")

,category,tickets,perc
0,Slow Data,8035,29.94
1,Dropped Call,6365,23.71
2,Network Coverage,4885,18.20
3,Billing,3738,13.93
4,Recharge Issue,2679,9.98
5,SIM/KYC,1139,4.24


In [44]:
# G2 · Does the complaint channel change how often a ticket is resolved?
# STEPS
# 1. Group complaints by channel.
# 2. Count.
# 3. Compute the resolved share as an average of is_resolved times 100.
q("""
SELECT channel, COUNT(*) AS tickets,
ROUND(AVG(is_resolved)*100,2) AS resolved_share
FROM complaints
GROUP BY channel
ORDER BY resolved_share DESC;
""")

,channel,tickets,resolved_share
0,Twitter/X,6698,70.36
1,Store,6799,70.33
2,Airtel App,6616,70.04
3,Call Center,6728,69.72


In [45]:
# G3 · How many tickets are still open?
# STEPS
# 1. Group by is_resolved.
# 2. Count and share.
q("""
 SELECT
    CASE WHEN is_resolved = 0 THEN 'Open' ELSE 'Resolved' END AS status,
    COUNT(*) AS tickets,
    ROUND(100.0 * COUNT() / (SELECT COUNT() FROM complaints), 2) AS share_pct
FROM complaints
GROUP BY is_resolved
""")

,status,tickets,share_pct
0,Open,8021,29.88
1,Resolved,18820,70.12


In [46]:
# Which complaint category is resolved least often, and most slowly?
# STEPS
# 1. Group by category.
# 2. Compute resolved share and average resolution time.
q("""
  SELECT category,
    COUNT(*) AS tickets,
    ROUND(AVG(is_resolved)*100,2) AS resolved_share,
    ROUND(AVG(resolution_days),2) AS avg_resolution_time
FROM complaints
GROUP BY category
""")

,category,tickets,resolved_share,avg_resolution_time
0,Billing,3738,70.87,7.50
1,Dropped Call,6365,69.71,7.46
2,Network Coverage,4885,69.52,7.60
3,Recharge Issue,2679,72.23,7.55
4,SIM/KYC,1139,68.57,7.65
5,Slow Data,8035,69.97,7.55


In [47]:
# 2 G5 · How many days does a resolved ticket take to close?
# STEPS
# 1. Filter to resolved tickets only.
# 2. Average, minimum and maximum resolution_days.
# 3. Say explicitly what population that is measured on.
q("""
SELECT
    COUNT(*) AS resolved_tickets,
    ROUND(AVG(resolution_days), 2) AS avg_resolution_days,
    MIN(resolution_days) AS min_resolution_days,
    MAX(resolution_days) AS max_resolution_days
FROM complaints
WHERE is_resolved = 1
""")

,resolved_tickets,avg_resolution_days,min_resolution_days,max_resolution_days
0,18820,7.54,1.0,14.0


In [48]:
# How many subscribers have raised 0, 1, 2 or more complaints?
# STEPS
# 1. Group customer_features by complaint_count.
# 2. Count subscribers at each level.
# 3. Read the whole distribution rather than the average.
q("""
SELECT complaint_count, COUNT(*) AS subs,
    ROUND(100.0 * COUNT() / (SELECT COUNT() FROM customer_features), 2) AS share_pct
FROM customer_features
GROUP BY complaint_count
""")

,complaint_count,subs,share_pct
0,0,6631,30.14
1,1,7854,35.70
2,2,4726,21.48
3,3,1926,8.75
4,4,625,2.84
5,5,183,0.83
6,6,44,0.20
7,7,10,0.05
8,8,1,0.00


In [49]:
# · Which five circles raise the most complaints per subscriber?
# STEPS
# 1. Group by circle_name.
# 2. Divide total tickets by subscribers so circles of different sizes compare.
# 3. Sort and take five
q("""
SELECT circle_name,
    COUNT(*) AS subs,
    ROUND(1.0 * SUM(complaint_count) / COUNT(*), 2) AS perc
    FROM customer_features
    GROUP BY circle_name
    ORDER BY perc DESC
    LIMIT 5;
""")

,circle_name,subs,perc
0,Tamil Nadu,731,1.29
1,Bihar,887,1.29
2,Punjab,558,1.27
3,Kolkata,572,1.27
4,Delhi,735,1.27


In [50]:
# H1 · How many of our current subscribers joined in each year?
# STEPS
# 1. Read customer_features.
# 2. Pull the year out of activation_date with strftime('%Y', …).
# 3. Group and count.
q("""
SELECT strftime('%Y', activation_date) AS year, COUNT(*) AS subs
FROM customer_features
GROUP BY year
ORDER BY year;
""")

,year,subs
0,2022,6922
1,2023,6925
2,2024,7053
3,2025,1100


In [51]:
# H2 · What share of each joining year has left?
# STEPS
# 1. Same grouping.
# 2. Add the churn count and rate.
# 3. Read down the churn column.
q("""
SELECT strftime('%Y', activation_date) AS year, COUNT(*) AS subs,
       SUM(is_churned) AS churned,
       ROUND(1.0 * SUM(is_churned) / COUNT(*), 2) AS churn_rate
FROM customer_features
GROUP BY year
ORDER BY year;
""")

,year,subs,churned,churn_rate
0,2022,6922,1374,0.20
1,2023,6925,1632,0.24
2,2024,7053,1901,0.27
3,2025,1100,319,0.29


In [52]:
# H3 · How does churn change with each completed year on the network?
# STEPS
# 1. Group by tenure_years
# 2. Sort by tenure
q("""
SELECT tenure_years, COUNT(*) AS subs,
         SUM(is_churned) AS churned,
         ROUND(1.0 * SUM(is_churned) / COUNT(*), 2) AS churn_rate
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,churned,churn_rate
0,0,4610,1328,0.29
1,1,6961,1725,0.25
2,2,7055,1541,0.22
3,3,3374,632,0.19


In [53]:
# H4 · How long had a leaver been with us when they left?
# STEPS
# 1. Filter to churned subscribers.
# 2. Average, minimum and maximum days_to_churn, which Step 2 built from the two dates.
q("""
SELECT AVG(days_to_churn) AS avg_days, MIN(days_to_churn) AS min_days, MAX(days_to_churn) AS max_days
FROM customer_features
WHERE is_churned = 1;
""")

,avg_days,min_days,max_days
0,567.183123,10,1259


In [54]:
# · Do longer-standing subscribers pay more?
# STEPS
# 1. Group by tenure_years.
# 2. Average both actual_arpu_6m and recharge_total_6m.
q("""
SELECT tenure_years, cOUNT(*) AS subs,
ROUND(AVG(actual_arpu_6m), 2) AS avg_arpu, 
ROUND(AVG(recharge_total_6m), 2) AS avg_recharge
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years;
""")

,tenure_years,subs,avg_arpu,avg_recharge
0,0,4610,329.10,1974.63
1,1,6961,342.72,2056.30
2,2,7055,350.13,2100.76
3,3,3374,357.22,2143.29


In [55]:
# 1 · How many subscribers have we lost, and what is the churn rate?
# S T E P S
# 1. Count all subscribers.
# 2. Sum is_churned.
# 3. Average it for the rate.
q("""
SELECT COUNT(*) AS subs,
       SUM(is_churned) AS churned_count,
       ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features;
""")

,subs,churned_count,churn_rate
0,22000,5226,23.75


In [56]:
# I2 · What is the churn rate for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Count, sum and average churn.
q("""
SELECT connection_type, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,churned,churn_rate
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


In [57]:
# I3 · Which six plans have the highest churn rate?
# STEPS
# 1. Group by plan_name, carrying plan_type.
# 2. Count and average churn.
# 3. Sort worst first, take six.
q("""
SELECT plan_name, plan_type, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY plan_name, plan_type
ORDER BY churn_rate DESC
LIMIT 6;
""")

,plan_name,plan_type,subs,churned,churn_rate
0,Prepaid Lite 99,Prepaid,996,310,31.12
1,Prepaid Smart 155,Prepaid,1137,343,30.17
2,Prepaid Entertainment 399,Prepaid,2142,562,26.24
3,Prepaid Quarterly 719,Prepaid,2561,662,25.85
4,Prepaid Value 209,Prepaid,1437,363,25.26
5,Prepaid Annual 1799,Prepaid,2901,729,25.13


In [58]:
# I4 · Which circles have the highest and the lowest churn rate?
# STEPS
# 1. Group by circle_name, carrying region.
#  2. Count and average churn.
# 3. Sort worst first and print all 22, because the spread is the finding.
q("""
SELECT circle_name, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY circle_name
ORDER BY churn_rate DESC;
""")

,circle_name,subs,churned,churn_rate
0,Himachal Pradesh,1247,417,33.44
1,Maharashtra,1428,470,32.91
2,Andhra Pradesh,1412,408,28.90
3,Mumbai,1395,392,28.10
4,Uttar Pradesh (East),554,151,27.26
5,Haryana,1041,265,25.46
6,Jammu & Kashmir,933,232,24.87
7,Gujarat,1210,298,24.63
8,Assam,1301,301,23.14
9,Uttar Pradesh (West),1081,248,22.94


In [59]:
# I5 · Do smartphone users leave at a different rate from feature-phone users?
# STEPS
# 1. Group by device_type.
# 2. Count and average churn.
# 3. Read the two numbers.
q("""
SELECT device_type, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY device_type;
""")

,device_type,subs,churned,churn_rate
0,Feature Phone,3029,719,23.74
1,Smartphone,18971,4507,23.76


In [60]:
# I6 · Does churn differ by acquisition channel?
# STEPS
# 1. Group by acquisition_channel.
# 2. Count and average churn.
# 3. Sort worst first.
q("""
SELECT acquisition_channel, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY acquisition_channel
ORDER BY churn_rate DESC;
""")

,acquisition_channel,subs,churned,churn_rate
0,Call Center,1770,436,24.63
1,Airtel App,6513,1576,24.20
2,Website,3081,743,24.12
3,Doorstep,1349,321,23.80
4,Retailer,9287,2150,23.15


In [61]:
# I7 · Does churn differ by age band?
# STEPS
# 1. Group by age_band.
# 2. Count and average churn.
# 3. Sort by band.
q("""
SELECT age_band, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY age_band
ORDER BY churn_rate DESC;
""")

,age_band,subs,churned,churn_rate
0,55+,1546,386,24.97
1,35-44,4824,1144,23.71
2,25-34,7467,1770,23.70
3,18-24,5268,1246,23.65
4,45-54,2895,680,23.49


In [62]:
# J1 · Does an open complaint make a subscriber more likely to leave?
# STEPS
# 1. Use CASE WHEN to build three groups from complaint_count and unresolved_count.
# 2. Count and average churn in each.
# 3. Sort so the three read in order.
q("""


SELECT CASE WHEN complaint_count=0
            THEN 'never complained'
            WHEN unresolved_count=0
            THEN 'all resolved'
            ELSE 'open ticket'
       END AS grp,
       COUNT(*) AS subs,
       ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY grp
ORDER BY grp ;

""")

,grp,subs,churn
0,all resolved,8689,23.75
1,never complained,6631,20.03
2,open ticket,6680,27.46


In [63]:
# · Does churn rise with each extra open ticket?
# S T E P S
# 1. Group by unresolved_count directly.
# 2. Count and average churn.
# 3. Read the gradient and then read the group sizes.
q("""
SELECT unresolved_count, COUNT(*) AS subs, ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY unresolved_count
ORDER BY unresolved_count ;
""")

,unresolved_count,subs,churn
0,0,15320,22.14
1,1,5515,26.07
2,2,1018,33.10
3,3,120,37.50
4,4,25,48.00
5,5,2,100.00


In [64]:
# K1 · What is the churn rate in each recency quartile?
# S T E P S
# 1. Group by r_score.
# 2. Average churn.
# 3. Look at it before saying anything.
q("""
SELECT r_score, COUNT(*) AS subs, ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY r_score
ORDER BY r_score
""")

,r_score,subs,churn
0,1,5500,91.49
1,2,5500,3.53
2,3,5500,0.00
3,4,5500,0.00


In [65]:
# K2 · What is the churn rate in each frequency quartile?
# STEPS
# 1. Group by f_score.
# 2. Average recharge_count to check the score means what it says.
# 3. Average churn.
q("""
SELECT f_score, COUNT(*) AS subs, ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY f_score
ORDER BY f_score
""")

,f_score,subs,churn
0,1,5500,45.44
1,2,5500,38.67
2,3,5500,7.76
3,4,5500,3.15


In [66]:
# L1 · What are the churn rate and the monthly revenue of each region?
# STEPS
# 1. Group by region.
# 2. Count, sum revenue and average churn.
# 3. Sort by churn.
q("""
SELECT region, COUNT(*) AS subs, SUM(actual_arpu_6m) AS total_revenue, ROUND(100.0*AVG(is_churned),2) AS churn_rate
FROM customer_features
GROUP BY region
ORDER BY churn_rate DESC
""")

,region,subs,total_revenue,churn_rate
0,West,4665,1571780.02,27.85
1,North,7231,2477493.21,23.73
2,East,5459,1893199.66,21.91
3,South,4645,1635737.25,21.85


In [67]:
# L2 · Which four circles churn the most, and what do their subscribers pay?
# STEPS
# 1. Group by circle and region.
# 2. Average churn and ARPU together.
# 3. Sort worst first, take four.
q("""
SELECT circle_name, region, COUNT(*) AS subs, SUM(actual_arpu_6m) AS total_revenue, ROUND(100.0*AVG(is_churned),2) AS churn_rate
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn_rate DESC
LIMIT 4
""")


,circle_name,region,subs,total_revenue,churn_rate
0,Himachal Pradesh,North,1247,390510.75,33.44
1,Maharashtra,West,1428,450032.30,32.91
2,Andhra Pradesh,South,1412,463369.80,28.90
3,Mumbai,West,1395,473288.16,28.10


In [68]:
# L7 · Do regions differ in revenue per subscriber and in data use?
# S T E P S
# 1. Group by region.
# 2. Average actual_arpu_6m and mb_per_active_day.
# 3. Sort by ARPU.
q("""
SELECT region, AVG(actual_arpu_6m) AS avg_arpu, AVG(mb_per_active_day) AS avg_data_use
FROM customer_features
GROUP BY region
ORDER BY avg_arpu DESC
""")

,region,avg_arpu,avg_data_use
0,South,352.150108,1430.498581
1,East,346.803382,1404.472713
2,North,342.621105,1388.546529
3,West,336.930337,1403.088643


In [69]:
# · How many subscribers are in group A and in group B?
# S T E P S
# 1. Group customer_features by ab_group.
# 2. Count and share.
q("""
SELECT ab_group, COUNT(*) AS subs
FROM customer_features
GROUP BY ab_group
""")

,ab_group,subs
0,A,10996
1,B,11004


In [70]:
# What is the churn rate in group A and in group B?
# STEPS
# 1. Group by ab_group.
# 2. Count, sum and average is_churned.
# 3. Read the two rates and then say nothing conclusive.
q("""
SELECT ab_group, COUNT(*) AS subs, SUM(is_churned) AS churned, ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
FROM customer_features
GROUP BY ab_group
""")

,ab_group,subs,churned,churn_rate
0,A,10996,2587,23.53
1,B,11004,2639,23.98


In [71]:
# Does revenue per subscriber differ between the groups, on either ARPU column?
# 1. Group by ab_group.
# 2. Average both the stated and the measured ARPU column.
# 3. Compare the two rows, and then compare the two columns.
q("""
SELECT ab_group, COUNT(*) AS subs, ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu, ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu
FROM customer_features
GROUP BY ab_group
""")

,ab_group,subs,avg_stated_arpu,avg_actual_arpu
0,A,10996,424.30,347.58
1,B,11004,418.99,341.35
